# Object-Oriented Programming in Python
### Professional Study Notes

| | |
|---|---|
| **Level** | Beginner → Advanced |
| **Python version** | 3.9+ (examples use 3.10+ features where noted) |
| **Format** | Concept → Example → Pitfalls → Practice |

---

## Table of Contents
1. [Introduction to OOP](#1.-Introduction-to-OOP)
2. [Classes and Objects](#2.-Classes-and-Objects)
3. [Instance vs Class Attributes](#3.-Instance-vs-Class-Attributes)
4. [Types of Methods](#4.-Types-of-Methods)
5. [Encapsulation](#5.-Encapsulation)
6. [Inheritance](#6.-Inheritance)
7. [Polymorphism](#7.-Polymorphism)
8. [Abstraction](#8.-Abstraction)
9. [Special (Dunder) Methods](#9.-Special-(Dunder)-Methods)
10. [Composition vs Inheritance](#10.-Composition-vs-Inheritance)
11. [Dataclasses](#11.-Dataclasses)
12. [Design Principles (SOLID)](#12.-Design-Principles-(SOLID))
13. [Common Pitfalls](#13.-Common-Pitfalls)
14. [Practice Exercises](#14.-Practice-Exercises)
15. [Quick Reference Cheat Sheet](#15.-Quick-Reference-Cheat-Sheet)

---
## 1. Introduction to OOP

**Object-Oriented Programming (OOP)** is a paradigm that organises software around **objects**, which bundle
**data (attributes)** and **behaviour (methods)** together, instead of around loose functions and global data.

### Why use OOP?
| Benefit | Meaning |
|---|---|
| **Modularity** | Code is split into self-contained units (classes). |
| **Reusability** | Inheritance and composition let you reuse existing code. |
| **Maintainability** | Changes stay local to one class. |
| **Scalability** | Large systems are easier to model with real-world concepts. |

### The Four Pillars of OOP
| Pillar | One-line definition |
|---|---|
| **Encapsulation** | Bundle data and methods; control access to internal state. |
| **Abstraction** | Expose only essential features; hide complexity. |
| **Inheritance** | A class acquires attributes and methods from another class. |
| **Polymorphism** | One interface, many implementations. |

> **Note:** Python is a *multi-paradigm* language. OOP is a tool, not a rule. Use it where modelling
> state + behaviour together makes the code clearer.

---
## 2. Classes and Objects

- A **class** is a blueprint that defines attributes and methods.
- An **object** (instance) is a concrete realisation of a class, created by calling the class.
- `__init__` is the **initialiser**, called automatically after an object is created.
- `self` refers to the current instance; Python passes it automatically as the first argument.

In [1]:
class Student:
    """Represents a student in a university system."""

    def __init__(self, name: str, roll_no: int, marks: float = 0.0):
        # Instance attributes: unique to each object
        self.name = name
        self.roll_no = roll_no
        self.marks = marks

    def is_passing(self, pass_mark: float = 40.0) -> bool:
        return self.marks >= pass_mark

    def __repr__(self) -> str:
        return f"Student(name={self.name!r}, roll_no={self.roll_no}, marks={self.marks})"


# Creating objects (instantiation)
s1 = Student("Aarav", 101, 82.5)
s2 = Student("Diya", 102, 35.0)

print(s1)
print(s2)
print("Aarav passing?", s1.is_passing())
print("Diya passing? ", s2.is_passing())
print("Type of s1:", type(s1))
print("Is s1 a Student?", isinstance(s1, Student))

Student(name='Aarav', roll_no=101, marks=82.5)
Student(name='Diya', roll_no=102, marks=35.0)
Aarav passing? True
Diya passing?  False
Type of s1: <class '__main__.Student'>
Is s1 a Student? True


### Key points
- `__init__` is **not** a constructor in the strict sense. `__new__` creates the object; `__init__` initialises it.
- Attributes can be added dynamically (`s1.city = "Indore"`), but defining them all in `__init__` is cleaner.
- Use `vars(obj)` or `obj.__dict__` to inspect an object's attributes.

In [2]:
print(vars(s1))
print(dir(s1)[-5:])   # last few names from dir()

{'name': 'Aarav', 'roll_no': 101, 'marks': 82.5}
['__weakref__', 'is_passing', 'marks', 'name', 'roll_no']


---
## 3. Instance vs Class Attributes

| | Instance attribute | Class attribute |
|---|---|---|
| Defined | Inside `__init__` via `self.x` | In the class body |
| Shared? | No, one copy per object | Yes, one copy for all objects |
| Typical use | Per-object state | Constants, counters, defaults |

In [3]:
class Employee:
    company = "TechNova"        # class attribute (shared)
    employee_count = 0          # class attribute used as a counter

    def __init__(self, name: str, salary: float):
        self.name = name        # instance attribute
        self.salary = salary
        Employee.employee_count += 1   # modify via the class, not self


e1 = Employee("Rohan", 50_000)
e2 = Employee("Sara", 65_000)

print(Employee.company, "|", e1.company, "|", e2.company)
print("Total employees:", Employee.employee_count)

# Pitfall: assigning via an instance creates a NEW instance attribute that shadows the class one
e1.company = "OtherCorp"
print(e1.company, "|", e2.company, "|", Employee.company)

TechNova | TechNova | TechNova
Total employees: 2
OtherCorp | TechNova | TechNova


---
## 4. Types of Methods

| Type | Decorator | First parameter | Can access | Typical use |
|---|---|---|---|---|
| **Instance method** | none | `self` | instance and class | Behaviour tied to one object |
| **Class method** | `@classmethod` | `cls` | class only | Alternative constructors, class-level logic |
| **Static method** | `@staticmethod` | none | neither | Utility function that belongs logically to the class |

In [4]:
from datetime import date


class Person:
    def __init__(self, name: str, birth_year: int):
        self.name = name
        self.birth_year = birth_year

    # Instance method
    def age(self) -> int:
        return date.today().year - self.birth_year

    # Class method used as an alternative constructor
    @classmethod
    def from_string(cls, text: str) -> "Person":
        name, year = text.split("-")
        return cls(name.strip(), int(year))

    # Static method: no access to self or cls
    @staticmethod
    def is_adult(age: int) -> bool:
        return age >= 18


p = Person.from_string("Meera - 2000")
print(p.name, p.birth_year)
print("Adult?", Person.is_adult(p.age()))

Meera 2000
Adult? True


> **Rule of thumb:** If a method needs the instance, make it an instance method. If it needs only the class
> (e.g. to build instances), use `@classmethod`. If it needs neither, consider whether it should be a plain
> module-level function.

---
## 5. Encapsulation

Encapsulation restricts direct access to an object's internal state and exposes a controlled interface.

### Access conventions in Python
Python has **no true private members**; it relies on naming conventions.

| Syntax | Meaning | Enforced? |
|---|---|---|
| `name` | Public | n/a |
| `_name` | Protected (internal use, by convention) | No |
| `__name` | Private via **name mangling** (`_ClassName__name`) | Partially |

In [5]:
class Demo:
    def __init__(self):
        self.public = "anyone can use me"
        self._protected = "internal use, please"
        self.__private = "name-mangled"


d = Demo()
print(d.public)
print(d._protected)

try:
    print(d.__private)
except AttributeError as err:
    print("AttributeError:", err)

print("Mangled access:", d._Demo__private)   # possible, but never do this in real code

anyone can use me
internal use, please
AttributeError: 'Demo' object has no attribute '__private'
Mangled access: name-mangled


### Properties: the Pythonic way to encapsulate
Use `@property` for validated, computed, or read-only attributes while keeping a clean attribute-style API.

In [6]:
class BankAccount:
    def __init__(self, owner: str, balance: float = 0.0):
        self.owner = owner
        self._balance = 0.0
        self.balance = balance          # goes through the setter, so it is validated

    @property
    def balance(self) -> float:
        """Current balance (read access)."""
        return self._balance

    @balance.setter
    def balance(self, value: float) -> None:
        if value < 0:
            raise ValueError("Balance cannot be negative")
        self._balance = value

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self._balance += amount

    def withdraw(self, amount: float) -> None:
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount


acct = BankAccount("Kabir", 1000)
acct.deposit(500)
acct.withdraw(200)
print("Balance:", acct.balance)

try:
    acct.balance = -50
except ValueError as err:
    print("Validation error:", err)

Balance: 1300
Validation error: Balance cannot be negative


### Read-only and computed properties
A property without a setter is read-only. A property can also compute its value on demand.

In [7]:
class Rectangle:
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    @property
    def area(self) -> float:          # computed, read-only
        return self.width * self.height


r = Rectangle(4, 5)
print("Area:", r.area)

try:
    r.area = 100
except AttributeError as err:
    print("AttributeError:", err)

Area: 20
AttributeError: property 'area' of 'Rectangle' object has no setter


---
## 6. Inheritance

Inheritance lets a **child (derived) class** reuse and extend a **parent (base) class**. It models an **"is-a"** relationship.

```
        Animal
       /      \
     Dog      Cat
```

### 6.1 Single inheritance and `super()`

In [8]:
class Animal:
    def __init__(self, name: str):
        self.name = name

    def speak(self) -> str:
        return f"{self.name} makes a sound"


class Dog(Animal):
    def __init__(self, name: str, breed: str):
        super().__init__(name)          # delegate initialisation to the parent
        self.breed = breed

    def speak(self) -> str:             # method overriding
        return f"{self.name} the {self.breed} barks"


class Cat(Animal):
    def speak(self) -> str:
        return f"{self.name} meows"


for pet in (Dog("Bruno", "Labrador"), Cat("Luna"), Animal("Generic")):
    print(pet.speak())

print(issubclass(Dog, Animal), isinstance(Dog("x", "y"), Animal))

Bruno the Labrador barks
Luna meows
Generic makes a sound
True True


### 6.2 Multilevel inheritance
A chain of inheritance: `Vehicle → Car → ElectricCar`.

In [9]:
class Vehicle:
    def __init__(self, brand: str):
        self.brand = brand

    def start(self) -> str:
        return f"{self.brand} vehicle started"


class Car(Vehicle):
    def __init__(self, brand: str, doors: int):
        super().__init__(brand)
        self.doors = doors


class ElectricCar(Car):
    def __init__(self, brand: str, doors: int, battery_kwh: float):
        super().__init__(brand, doors)
        self.battery_kwh = battery_kwh

    def start(self) -> str:
        return super().start() + " silently (electric)"


ev = ElectricCar("Tata", 4, 40.5)
print(ev.start())
print(ElectricCar.__mro__)

Tata vehicle started silently (electric)
(<class '__main__.ElectricCar'>, <class '__main__.Car'>, <class '__main__.Vehicle'>, <class 'object'>)


### 6.3 Multiple inheritance and the Method Resolution Order (MRO)
Python supports multiple inheritance. When a method is looked up, Python follows the **MRO**, computed by the
**C3 linearisation** algorithm. Inspect it with `Class.__mro__` or `Class.mro()`.

The *diamond problem* is handled cleanly because `super()` follows the MRO.

In [10]:
class A:
    def hello(self):
        return "A.hello"


class B(A):
    def hello(self):
        return "B.hello -> " + super().hello()


class C(A):
    def hello(self):
        return "C.hello -> " + super().hello()


class D(B, C):                     # diamond: D -> B -> C -> A
    def hello(self):
        return "D.hello -> " + super().hello()


print(D().hello())
print([cls.__name__ for cls in D.__mro__])

D.hello -> B.hello -> C.hello -> A.hello
['D', 'B', 'C', 'A', 'object']


### 6.4 Mixins
A **mixin** is a small class that provides one focused piece of behaviour and is meant to be combined with others,
not instantiated on its own.

In [11]:
import json


class JSONMixin:
    def to_json(self) -> str:
        return json.dumps(self.__dict__)


class LoggingMixin:
    def log(self, message: str) -> None:
        print(f"[{self.__class__.__name__}] {message}")


class User(JSONMixin, LoggingMixin):
    def __init__(self, username: str, email: str):
        self.username = username
        self.email = email


u = User("neha", "neha@example.com")
u.log("user created")
print(u.to_json())

[User] user created
{"username": "neha", "email": "neha@example.com"}


---
## 7. Polymorphism

Polymorphism ("many forms") lets different classes be used through the same interface.

| Form | How Python achieves it |
|---|---|
| **Method overriding** | A subclass redefines a parent method. |
| **Duck typing** | "If it walks like a duck and quacks like a duck, it is a duck." Only behaviour matters, not type. |
| **Operator overloading** | Dunder methods such as `__add__` give operators custom meaning. |

In [12]:
import math


class Shape:
    def area(self) -> float:
        raise NotImplementedError("Subclasses must implement area()")


class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        return math.pi * self.radius ** 2


class Square(Shape):
    def __init__(self, side: float):
        self.side = side

    def area(self) -> float:
        return self.side ** 2


# Polymorphism: the same call works for every Shape
shapes = [Circle(3), Square(4)]
for s in shapes:
    print(f"{type(s).__name__:<8} area = {s.area():.2f}")

Circle   area = 28.27
Square   area = 16.00


### Duck typing
No common base class is required. Any object with the right method works.

In [13]:
class Duck:
    def speak(self):
        return "Quack"


class Robot:
    def speak(self):
        return "Beep boop"


def make_it_speak(thing):
    # No isinstance check: we only care that .speak() exists
    print(thing.speak())


for obj in (Duck(), Robot()):
    make_it_speak(obj)

Quack
Beep boop


---
## 8. Abstraction

Abstraction hides implementation details and defines **what** an object does, not **how**.
In Python, use the `abc` module to create **Abstract Base Classes (ABCs)**.

- A class with one or more `@abstractmethod`s **cannot be instantiated**.
- Subclasses **must** implement all abstract methods.

In [14]:
from abc import ABC, abstractmethod


class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount: float) -> str:
        """Process a payment."""

    def receipt(self, amount: float) -> str:      # concrete method shared by all subclasses
        return f"Receipt: paid {amount:.2f} using {self.__class__.__name__}"


class CreditCard(PaymentMethod):
    def pay(self, amount: float) -> str:
        return f"Charged {amount:.2f} to credit card"


class UPI(PaymentMethod):
    def pay(self, amount: float) -> str:
        return f"Sent {amount:.2f} via UPI"


try:
    PaymentMethod()
except TypeError as err:
    print("TypeError:", err)

for method in (CreditCard(), UPI()):
    print(method.pay(1500))
    print(method.receipt(1500))

TypeError: Can't instantiate abstract class PaymentMethod without an implementation for abstract method 'pay'
Charged 1500.00 to credit card
Receipt: paid 1500.00 using CreditCard
Sent 1500.00 via UPI
Receipt: paid 1500.00 using UPI


---
## 9. Special (Dunder) Methods

Dunder ("double underscore") methods let your objects integrate with Python's built-in syntax and functions.

| Category | Methods | Triggered by |
|---|---|---|
| Representation | `__str__`, `__repr__` | `print(obj)`, `repr(obj)` |
| Comparison | `__eq__`, `__lt__`, `__le__`, ... | `==`, `<`, `<=` |
| Arithmetic | `__add__`, `__sub__`, `__mul__` | `+`, `-`, `*` |
| Container | `__len__`, `__getitem__`, `__contains__`, `__iter__` | `len()`, `obj[i]`, `in`, `for` |
| Callable | `__call__` | `obj()` |
| Context manager | `__enter__`, `__exit__` | `with obj:` |
| Hashing | `__hash__` | `set`, `dict` keys |

> `__repr__` should be **unambiguous** (for developers); `__str__` should be **readable** (for users).
> If only one is defined, define `__repr__`.

In [15]:
from functools import total_ordering


@total_ordering                      # generates the other comparisons from __eq__ and __lt__
class Money:
    def __init__(self, amount: float, currency: str = "INR"):
        self.amount = amount
        self.currency = currency

    def __repr__(self):
        return f"Money({self.amount!r}, {self.currency!r})"

    def __str__(self):
        return f"{self.currency} {self.amount:,.2f}"

    def __add__(self, other):
        if not isinstance(other, Money) or other.currency != self.currency:
            return NotImplemented
        return Money(self.amount + other.amount, self.currency)

    def __eq__(self, other):
        return isinstance(other, Money) and (self.amount, self.currency) == (other.amount, other.currency)

    def __lt__(self, other):
        return self.amount < other.amount

    def __hash__(self):
        return hash((self.amount, self.currency))

    def __bool__(self):
        return self.amount != 0


a, b = Money(1500), Money(2500.5)
print(a + b)
print(repr(a))
print(a < b, a >= b, a == Money(1500))
print(sorted([b, a]))
print("Truthy?", bool(Money(0)))

INR 4,000.50
Money(1500, 'INR')
True False True
[Money(1500, 'INR'), Money(2500.5, 'INR')]
Truthy? False


### Making a class iterable and container-like

In [16]:
class Playlist:
    def __init__(self, *songs):
        self._songs = list(songs)

    def __len__(self):
        return len(self._songs)

    def __getitem__(self, index):
        return self._songs[index]

    def __contains__(self, song):
        return song in self._songs

    def __iter__(self):
        return iter(self._songs)


pl = Playlist("Song A", "Song B", "Song C")
print(len(pl), pl[0], pl[-1], "Song B" in pl)
for song in pl:
    print(" -", song)

3 Song A Song C True
 - Song A
 - Song B
 - Song C


### Context managers (`with` statement)

In [17]:
import time


class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.elapsed = time.perf_counter() - self.start
        print(f"Elapsed: {self.elapsed:.4f}s")
        return False          # do not suppress exceptions


with Timer():
    sum(range(1_000_000))

Elapsed: 0.0131s


---
## 10. Composition vs Inheritance

| | Inheritance | Composition |
|---|---|---|
| Relationship | **is-a** | **has-a** |
| Coupling | Tight | Loose |
| Flexibility | Fixed at class definition | Can be changed at runtime |
| Risk | Fragile base class, deep hierarchies | More wiring code |

> **Guideline: "Favour composition over inheritance."** Use inheritance only for a genuine *is-a* relationship
> where the subclass can be substituted for the parent everywhere.

In [18]:
class Engine:
    def start(self) -> str:
        return "Engine started"


class GPS:
    def locate(self) -> str:
        return "Location: 22.7196 N, 75.8577 E"


class Car:
    """A Car HAS an Engine and HAS a GPS. It is not an Engine."""

    def __init__(self, engine: Engine, gps: GPS):
        self.engine = engine
        self.gps = gps

    def drive(self) -> str:
        return f"{self.engine.start()} | {self.gps.locate()}"


print(Car(Engine(), GPS()).drive())

Engine started | Location: 22.7196 N, 75.8577 E


---
## 11. Dataclasses

`@dataclass` (Python 3.7+) auto-generates `__init__`, `__repr__`, `__eq__` and more, ideal for classes that mostly hold data.

| Option | Effect |
|---|---|
| `frozen=True` | Immutable instances (also hashable) |
| `order=True` | Generates `<`, `<=`, `>`, `>=` |
| `slots=True` | Lower memory, faster attribute access (3.10+) |
| `field(default_factory=list)` | Safe mutable defaults |

In [19]:
from dataclasses import dataclass, field


@dataclass(order=True)
class Product:
    name: str
    price: float
    tags: list[str] = field(default_factory=list, compare=False)

    def discounted(self, percent: float) -> float:
        return round(self.price * (1 - percent / 100), 2)

    def __post_init__(self):                     # runs after the generated __init__
        if self.price < 0:
            raise ValueError("price must be non-negative")


p1 = Product("Keyboard", 1999.0, ["usb", "mechanical"])
p2 = Product("Mouse", 799.0)

print(p1)
print(p1 == Product("Keyboard", 1999.0), p2 < p1)
print("After 10% off:", p1.discounted(10))


@dataclass(frozen=True)
class Point:
    x: float
    y: float


pt = Point(1, 2)
try:
    pt.x = 10
except Exception as err:
    print(type(err).__name__, "-", err)

Product(name='Keyboard', price=1999.0, tags=['usb', 'mechanical'])
True False
After 10% off: 1799.1
FrozenInstanceError - cannot assign to field 'x'


---
## 12. Design Principles (SOLID)

| Letter | Principle | Meaning |
|---|---|---|
| **S** | Single Responsibility | A class should have one reason to change. |
| **O** | Open/Closed | Open for extension, closed for modification. |
| **L** | Liskov Substitution | Subclasses must be usable wherever the parent is expected. |
| **I** | Interface Segregation | Prefer small, focused interfaces over large ones. |
| **D** | Dependency Inversion | Depend on abstractions, not concrete classes. |

### Example: Open/Closed + Dependency Inversion
Adding a new notifier requires **no change** to `OrderService`.

In [20]:
from abc import ABC, abstractmethod


class Notifier(ABC):
    @abstractmethod
    def send(self, message: str) -> None: ...


class EmailNotifier(Notifier):
    def send(self, message: str) -> None:
        print(f"[EMAIL] {message}")


class SMSNotifier(Notifier):
    def send(self, message: str) -> None:
        print(f"[SMS] {message}")


class OrderService:
    def __init__(self, notifier: Notifier):      # depends on the abstraction
        self.notifier = notifier

    def place_order(self, item: str) -> None:
        # ... business logic ...
        self.notifier.send(f"Order placed for {item}")


OrderService(EmailNotifier()).place_order("Laptop")
OrderService(SMSNotifier()).place_order("Phone")

[EMAIL] Order placed for Laptop
[SMS] Order placed for Phone


### Liskov Substitution: a classic violation
A `Square` that inherits from `Rectangle` breaks expectations: setting the width silently changes the height.
Prefer a common abstraction (`Shape`) over forcing an incorrect *is-a* relationship.

---
## 13. Common Pitfalls

### 13.1 Mutable default arguments
Default values are evaluated **once**, at function definition time.

In [21]:
class BadCart:
    def __init__(self, items=[]):          # BAD: the same list is shared by all instances
        self.items = items


class GoodCart:
    def __init__(self, items=None):        # GOOD
        self.items = items if items is not None else []


a, b = BadCart(), BadCart()
a.items.append("apple")
print("BadCart  b.items:", b.items)        # surprise: ['apple']

c, d = GoodCart(), GoodCart()
c.items.append("apple")
print("GoodCart d.items:", d.items)        # []

BadCart  b.items: ['apple']
GoodCart d.items: []


### 13.2 Mutable class attributes shared across instances

In [22]:
class Team:
    members = []                 # BAD: shared by every Team

    def add(self, name):
        self.members.append(name)


t1, t2 = Team(), Team()
t1.add("Alice")
print(t2.members)                # ['Alice'], not what you wanted

['Alice']


### 13.3 Forgetting `super().__init__()`
If a child defines `__init__` and does not call the parent's, parent attributes are never set.

### 13.4 Overusing inheritance
Deep hierarchies are hard to understand and change. Prefer composition and small, focused classes.

### 13.5 Defining `__eq__` without `__hash__`
Defining `__eq__` sets `__hash__` to `None`, making instances unhashable (unusable in sets or as dict keys)
unless you also define `__hash__`. Dataclasses with `frozen=True` handle this for you.

### 13.6 Getters/setters everywhere
Don't write `get_x()` / `set_x()` by default. Start with a plain public attribute and switch to `@property`
later if validation is needed; the external API does not change.

---
## 14. Practice Exercises

Try each exercise before opening the solution.

| # | Exercise | Concepts |
|---|---|---|
| 1 | Build a `Library` that stores `Book` objects, supports `add`, `remove`, and search by author. | Classes, composition |
| 2 | Create an abstract `Employee` with `Manager` and `Developer` subclasses that compute salary differently. | ABC, polymorphism |
| 3 | Implement a `Vector2D` class supporting `+`, `-`, scalar `*`, `abs()` and `==`. | Dunder methods |
| 4 | Create a `Temperature` class storing Celsius with a computed `fahrenheit` property (with setter). | Properties |
| 5 | Build a `Stack` class with `push`, `pop`, `peek`, `__len__`, and `__iter__`. | Container protocol |

### Exercise 3: Solution

In [23]:
import math


class Vector2D:
    def __init__(self, x: float, y: float):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vector2D({self.x}, {self.y})"

    def __add__(self, other):
        return Vector2D(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        return Vector2D(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar: float):
        return Vector2D(self.x * scalar, self.y * scalar)

    __rmul__ = __mul__                 # allows 3 * v

    def __abs__(self):
        return math.hypot(self.x, self.y)

    def __eq__(self, other):
        return isinstance(other, Vector2D) and (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))


v1, v2 = Vector2D(3, 4), Vector2D(1, 2)
print(v1 + v2, v1 - v2, v1 * 2, 3 * v1)
print(abs(v1), v1 == Vector2D(3, 4))

Vector2D(4, 6) Vector2D(2, 2) Vector2D(6, 8) Vector2D(9, 12)
5.0 True


### Exercise 4: Solution

In [24]:
class Temperature:
    def __init__(self, celsius: float = 0.0):
        self.celsius = celsius

    @property
    def fahrenheit(self) -> float:
        return self.celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value: float) -> None:
        self.celsius = (value - 32) * 5 / 9


t = Temperature(100)
print(t.fahrenheit)
t.fahrenheit = 32
print(t.celsius)

212.0
0.0


---
## 15. Quick Reference Cheat Sheet

```python
class ClassName(Parent):                 # inheritance
    class_attr = 0                       # shared by all instances

    def __init__(self, x):               # initialiser
        super().__init__()               # call parent
        self.x = x                       # instance attribute
        self._internal = None            # protected (convention)
        self.__secret = None             # name-mangled

    @property
    def value(self): ...                 # getter
    @value.setter
    def value(self, v): ...              # setter

    @classmethod
    def create(cls, ...): ...            # alternative constructor
    @staticmethod
    def helper(...): ...                 # utility

    def __repr__(self): ...              # developer representation
    def __str__(self): ...               # user representation
```

| Task | Tool |
|---|---|
| Check type / relationship | `isinstance(obj, C)`, `issubclass(A, B)` |
| Inspect attributes | `vars(obj)`, `dir(obj)`, `hasattr`, `getattr`, `setattr` |
| Inspect inheritance | `C.__mro__`, `C.__bases__` |
| Force subclass implementation | `ABC` + `@abstractmethod` |
| Data-holder class | `@dataclass` |
| Memory-efficient class | `__slots__` or `@dataclass(slots=True)` |

### Best-practice checklist
- [ ] Each class has a single, clear responsibility.
- [ ] Attributes are initialised in `__init__`.
- [ ] `__repr__` is defined for debugging.
- [ ] Inheritance is used only for true *is-a* relationships; otherwise compose.
- [ ] Mutable defaults are avoided (`None` or `default_factory`).
- [ ] Public API is small; internals are prefixed with `_`.
- [ ] Type hints and docstrings are present.
- [ ] Classes depend on abstractions, not concrete implementations.

---
*End of notes. Revisit the exercises, then explore: `__slots__`, descriptors, metaclasses, `typing.Protocol`, and design patterns (Factory, Strategy, Observer, Singleton).*